In [3]:
from langgraph.graph import StateGraph, START, END
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from typing import TypedDict

In [2]:
load_dotenv()

model = ChatOpenAI()

In [18]:
class BlogState(TypedDict):
    title: str
    outline: str
    blog: str
    evaluation_score: str

In [12]:
def create_outline(state: BlogState) -> BlogState:

    # extract title from state
    title = state['title']

    # generate prompt to create outline
    prompt = f'Generate a short outline for a blog post with the title: {title}'

    # call llm with prompt
    outline = model.invoke(prompt)

    state['outline'] = outline

    return state

In [13]:
def create_blog(state: BlogState) -> BlogState:

    # extract title from state
    title = state['title']
    outline = state['outline']

    # generate prompt to create blog post
    prompt = f'Generate a short blog with the title - {title} and outline - {outline}'

    # call llm with prompt
    blog = model.invoke(prompt)

    state['blog'] = blog

    return state

In [19]:
def evaluate_blog(state: BlogState) -> BlogState:

    #extract required data from state
    title = state['title']
    outline = state['outline']
    blog = state['blog']

    # create a prompt
    prompt = f'Evaluate the blog with the title - {title}, outline - {outline} and blog - {blog}. Give a score from 1 to 10'

    # send prompt to llm
    evaluation_score = model.invoke(prompt)

    state['evaluation_score'] = evaluation_score

    return state

In [20]:
# define graph
graph = StateGraph(BlogState)

# add nodes
graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

# add edges
graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

#compile graph
workflow = graph.compile()

In [21]:
initial_state = {'title': 'Need for beach cleanups'}

final_state = workflow.invoke(initial_state)

print(final_state['evaluation_score'])

content='I would give this blog a score of 9 out of 10. It covers all the essential aspects of the need for beach cleanups, including the explanation of pollution levels, statistics on beach pollution, benefits of cleanups, how to organize a cleanup, success stories, and a compelling call to action. The content is well-structured, informative, and effectively conveys the importance of beach cleanups in a clear and persuasive manner. The only room for improvement would be to expand on some of the statistics and success stories to provide even more compelling evidence for the need for beach cleanups.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 119, 'prompt_tokens': 1131, 'total_tokens': 1250, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider':